In [ ]:
import glob
import re
import os

import numpy as np
from pathlib import Path

from pymor.basic import *
from pymor.core.pickle import load

from RBInvParam.problems.elasticity.build import build_InstationaryModelIP

set_log_levels({
    'pymor' : 'WARN'
})

set_defaults({})


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt

fontsize = 14
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "cm",
    "font.size": fontsize,
    #'text.latex.preamble': r'\usepackage{amsfonts} \usepackage{accents} \usepackage{mathrsfs} \usepackage{bm}',
    'text.latex.preamble': r'\usepackage{bm}',
    'figure.dpi': 200
})

In [ ]:
from typing import Dict, Tuple, Optional

def get_last_file(path: Path) -> Path | None:
    # --- Step 1: Look for final files first ---
    final_candidates = [
        path / "TR_IRGNM_final.pkl",
        path / "FOM_IRGNM_final.pkl"
    ]
    
    for final_file in final_candidates:
        if final_file.exists():
            return final_file.name  # Return immediately if found
    
    # --- Step 2: If no final file exists, find the highest index file ---
    files = glob.glob(os.path.join(path, "TR_IRGNM_*.pkl"))
    files += glob.glob(os.path.join(path, "FOM_IRGNM_*.pkl"))

    indexed_files = []
    for f in files:
        match = re.search(r'(?:TR|FOM)_IRGNM_(\d+)\.pkl$', os.path.basename(f))
        if match:
            idx = int(match.group(1))
            indexed_files.append((idx, f))

    if indexed_files:
        _, max_file = max(indexed_files, key=lambda x: x[0])
        return Path(max_file).name

    print("No matching IRGNM result files found.")
    return None

def filter_and_reorder(d: Dict, pattern: str = r'.*FOM.*') -> Tuple[Dict, Optional[str]]:
    regex = re.compile(pattern)
    matching = [k for k in d if regex.search(k)]
    non_matching = [k for k in d if k not in matching]

    # Sort keys alphabetically within each group
    non_matching_sorted = sorted(non_matching)
    matching_sorted = sorted(matching)

    # Build the reordered dict: alphabetically sorted non-matching first, then matching ones
    reordered = {k: d[k] for k in non_matching_sorted}
    reordered.update({k: d[k] for k in matching_sorted})

    # Return reordered dict and the single matching key (if exactly one match)
    if len(matching_sorted) == 1:
        return reordered, matching_sorted[0]
    else:
        return reordered, None

In [ ]:
SAVE_PATH = Path('/home/dealii/workdir/figs')
#SAVE_PATH = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/figs')

########################################################################################

#WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')
WORK_DIR = Path('/home/dealii/workdir/experiments')
# WORK_DIR = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/experiments')

subset = 'elasticity'
obs_op = 'sensors'
#data_dir_path = WORK_DIR / 'elasticity_alu' / 'elasticity_alu_new_CG_1e-12'  / 'elasticity_alu_noise_level'
data_dir_path = WORK_DIR / 'elasticity_alu_non_normalize' / 'elasticity_alu_2'




experiment_names = []
#pattern = re.compile(rf'^.*_{obs_op}')
pattern = re.compile(rf'^.*')


#pattern = re.compile(rf'^.*')

experiment_names += [
    d.name for d in data_dir_path.iterdir()
    if d.is_dir() and pattern.match(d.name)
]

data_paths = [data_dir_path / experiment_name for experiment_name in experiment_names]
file_names = [get_last_file(data_path) for data_path in data_paths]              

########################################################################################

# data_dir_path = Path('/home/dealii/workdir/examples/hyperelasticity/dumps') / '20260322_183605_TR_IRGNM'
# #data_dir_path = Path('/home/benedikt/Dokumente/parabolische_inverse_probleme/examples/hyperelasticity/dumps') / '20260320_163140_TR_IRGNM'

# experiment_names = []
# pattern = re.compile(rf'.*')

# experiment_names += [
#     d.name for d in data_dir_path.iterdir()
#     if d.is_dir() and pattern.match(d.name)
# ]

# data_paths = [data_dir_path]
# file_names = [get_last_file(data_path) for data_path in data_paths]

########################################################################################

setup = None
data = {}
optimizer_parameters = {}

for (data_path, file_name) in zip(data_paths, file_names):
    try:
        with open(data_path / file_name, 'rb') as file:
            data_ = load(file)
        data[str(data_path.name)] = data_
    except TypeError:
        print(f"Can not find dumps for {data_path}")
    except:
        print(f"Can not open {data_path / file_name}")

    if not setup:
        with open(data_path / 'setup.pkl', 'rb') as file:
            setup = load(file)

    
    optimizer_parameter_path = data_path / 'optimizer_parameter.pkl'
    with open(optimizer_parameter_path, 'rb') as file:
        optimizer_parameter = load(file)
        
    optimizer_parameters[str(data_path.name)] = optimizer_parameter

data, FOM_key = filter_and_reorder(data, pattern=r'.*FOM.*')
print(experiment_names)


In [ ]:
# q_exact_hard = setup['q_exact']
# #q_exact_soft = setup['q_exact']

q_exact_1 = setup['q_exact']


q_exact_2 = setup['q_exact']


q_exact_3 = setup['q_exact']

In [ ]:
experiment_names

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

from matplotlib.ticker import FormatStrFormatter
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable  # <<< NEW

plt.rcParams.update({"font.size": 16})

title_pad = 7
time_step = 0
time_dep = False
idx = -1

# FOM_keys = [
#     'elasticity_alu_large_patch_FOM_identity',
#     'elasticity_alu_large_patch_FOM_grid',
#     'elasticity_alu_large_patch_FOM_sensors',
# ]

# TR_keys = [
#     'elasticity_alu_large_patch_TR_identity',
#     'elasticity_alu_large_patch_TR_grid',
#     'elasticity_alu_large_patch_TR_sensors',
# ]


FOM_keys = [
    'elasticity_alu_FOM_identity',
    'elasticity_alu_FOM_grid',
    'elasticity_alu_FOM_sensors',
]

TR_keys = [
    'elasticity_alu_TR_identity',
    'elasticity_alu_TR_grid',
    'elasticity_alu_TR_sensors',
]




assert len(FOM_keys) == len(TR_keys)

# We only create axes for the images now (3 per row); colorbars will be attached later
fig = plt.figure(figsize=(10, len(TR_keys) * 3), constrained_layout=True)

gs = gridspec.GridSpec(
    len(TR_keys), 3,          # 3 image axes per row
    figure=fig,
    wspace=0.10
)

ax = []

for i in range(len(TR_keys)):
    ax0 = fig.add_subplot(gs[i, 0])
    ax1 = fig.add_subplot(gs[i, 1])
    ax2 = fig.add_subplot(gs[i, 2])

    ax.append([ax0, ax1, ax2])

# --- build grid (assuming q_* are defined on a square grid) ---
x = np.arange(-30, 31)
y = np.arange(-30, 31)
X, Y = np.meshgrid(x, y)

# x = np.arange(-15, 16)
# y = np.arange(-15, 16)
# X, Y = np.meshgrid(x, y)

# titles = [
#     r'$\bm{q}^{\textrm{\footnotesize FOM}}$',
#     r'$\bm{q}^{\textrm{\footnotesize TR}}$',
#     r'$\bm{q}^{\textrm{\footnotesize FOM}} - \bm{q}^{\textrm{\footnotesize TR}}$'
# ]

titles = [
    r'$q^{\textrm{\footnotesize FOM}}$',
    r'$q^{\textrm{\footnotesize TR}}$',
    r'$q^{\textrm{\footnotesize FOM}} - q^{\textrm{\footnotesize TR}}$'
]

experiment_name = [
    '1',
    '2(a)',
    '2(b)'
]


extent = [-0.5, 0.5, -0.5, 0.5]
ticks = [-0.5, 0.0, 0.5]


fig.suptitle('Point defect I')
#fig.suptitle('Large Inclusion II')
# --- Plot with imshow + perfectly aligned colorbars ---
for i in range(len(TR_keys)):

    q_exact = setup['q_exact'][time_step, :]
    q_FOM   = data[FOM_keys[i]]['q'][idx].to_numpy()[time_step, :]
    q_TR    = data[TR_keys[i]]['q'][idx].to_numpy()[time_step, :]

    # reshape
    nx, ny = X.shape
    q_exact = q_exact.reshape(nx, ny)
    q_FOM   = q_FOM.reshape(nx, ny)
    q_TR    = q_TR.reshape(nx, ny)

    datasets = [q_FOM, q_TR, q_FOM - q_TR]
    vmin_shared = min(datasets[0].min(), datasets[1].min())
    vmax_shared = max(datasets[0].max(), datasets[1].max())

    for j, (ax_, q, title) in enumerate(zip(ax[i], datasets, titles)):
        #ax_.set_title(experiment_name[i] + ': ' + title, pad=title_pad)
        
        # if i == 0:
        #     ax_.set_title(experiment_name[i] + ': ' + title, pad=title_pad)
        # else:
        #     ax_.set_title(experiment_name[i], pad=title_pad)

        if i == 0:
            ax_.set_title(title, pad=title_pad)
        
        if j == 0:
            ax_.text(
                -0.45, 0.5, experiment_name[i],
                transform=ax_.transAxes,
                rotation=90,
                va='center', ha='center'
            )
        # shared range for first two plots, individual for the difference plot
        if j < 2:  # 0: FOM, 1: TR
            vmin, vmax = vmin_shared, vmax_shared
        else:      # 2: FOM - TR
            vmin, vmax = q.min(), q.max()

        im = ax_.imshow(
            q,
            origin="lower",
            #extent=[x.min(), x.max(), y.min(), y.max()],
            extent=extent,
            cmap="viridis",
            vmin=vmin,
            vmax=vmax,
            aspect="equal",
            interpolation='bilinear'
        )
        
        ax_.set_xticks(ticks)
        ax_.set_yticks(ticks)


        if i == len(TR_keys) - 1:
            ax_.set_xlabel(r'$z\;[\mathrm{m}]$')
        else:
            ax_.tick_params(axis='x', labelbottom=False)
        
        # only left column gets y-labels
        if j == 0:
            ax_.set_ylabel(r'$y\;[\mathrm{m}]$')
        else:
            ax_.tick_params(axis='y', labelleft=False)
            
        # --- attach a colorbar axis with same height as ax_ ---
        divider = make_axes_locatable(ax_)
        cax_ = divider.append_axes("right", size="3%", pad=0.10)  # size and pad adjustable

        cb = fig.colorbar(im, cax=cax_)
        cb.ax.tick_params(labelsize=10, pad=2)
        cb.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

# --- Show / save ---
#fig.tight_layout()
#plt.show()
fig.savefig(SAVE_PATH / Path('point_defects_2.pdf'), bbox_inches="tight")


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

from matplotlib.ticker import FormatStrFormatter
from pathlib import Path
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcParams.update({"font.size": 16})

title_pad = 7
time_step = 0
time_dep = False
idx = -1

FOM_keys = [
    'elasticity_alu_large_patch_FOM_identity',
    'elasticity_alu_large_patch_FOM_grid',
    'elasticity_alu_large_patch_FOM_sensors',
]

TR_keys = [
    'elasticity_alu_large_patch_TR_identity',
    'elasticity_alu_large_patch_TR_grid',
    'elasticity_alu_large_patch_TR_sensors',
]

assert len(FOM_keys) == len(TR_keys)

titles = [
    r'$q^{\textrm{\footnotesize FOM}}$',
    r'$q^{\textrm{\footnotesize TR}}$',
    r'$q^{\textrm{\footnotesize FOM}} - q^{\textrm{\footnotesize TR}}$'
]

experiment_name = [
    '1',
    '2(a)',
    '2(b)'
]

# --- build grid (assuming q_* are defined on a square grid) ---
x = np.arange(-30, 31)
y = np.arange(-30, 31)
X, Y = np.meshgrid(x, y)

extent = [-0.5, 0.5, -0.5, 0.5]
ticks = [-0.5, 0.0, 0.5]

# =============================================================================
# Figure 2: only second column (TR), horizontal layout
# =============================================================================
fig_tr = plt.figure(figsize=(10, 3.2), constrained_layout=True)

gs_tr = gridspec.GridSpec(
    1, len(TR_keys),
    figure=fig_tr,
    wspace=0.10
)

ax_tr = []
for i in range(len(TR_keys)):
    ax_tr.append(fig_tr.add_subplot(gs_tr[0, i]))

# shared color scale over all TR plots
all_q_TR = []
for i in range(len(TR_keys)):
    q_TR = data[TR_keys[i]]['q'][idx].to_numpy()[time_step, :]
    all_q_TR.append(q_TR.reshape(nx, ny))

vmin_tr = min(q.min() for q in all_q_TR)
vmax_tr = max(q.max() for q in all_q_TR)

for i in range(len(TR_keys)):
    q_TR = data[TR_keys[i]]['q'][idx].to_numpy()[time_step, :]
    q_TR = q_TR.reshape(nx, ny)

    ax_ = ax_tr[i]
    im = ax_.imshow(
        q_TR,
        origin="lower",
        extent=extent,
        cmap="viridis",
        vmin=vmin_tr,
        vmax=vmax_tr,
        aspect="equal",
        interpolation='bilinear'
    )

    ax_.set_title(experiment_name[i] + ': ' + r'$q^{\textrm{\footnotesize TR}}$', pad=title_pad)
    ax_.set_xticks(ticks)
    ax_.set_yticks(ticks)
    ax_.set_xlabel(r'$z\;[\mathrm{m}]$')

    if i == 0:
        ax_.set_ylabel(r'$y\;[\mathrm{m}]$')
    else:
        ax_.tick_params(axis='y', labelleft=False)

    divider = make_axes_locatable(ax_)
    cax_ = divider.append_axes("right", size="3%", pad=0.10)

    cb = fig_tr.colorbar(im, cax=cax_)
    cb.ax.tick_params(labelsize=10, pad=2)
    cb.ax.yaxis.set_major_formatter(FormatStrFormatter('%.2f'))

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

setups = [
    q_exact_2, 
    q_exact_1,
    q_exact_3
]
titles = [
    'I', 
    'II',
    'III'
]   # or ['(I)', '(II)']

# setups = [
#     q_exact_hard, 
#     q_exact_soft, 
# ]
# titles = [
#     'I', 
#     'II',
# ]   # or ['(I)', '(II)']

n = 61
extent = [-0.5, 0.5, -0.5, 0.5]
ticks = [-0.5, 0.0, 0.5]

if len(setups) == 2:
    fig, ax = plt.subplots(
        1, 2,
        figsize=(7.2, 3.2),
        constrained_layout=False,
        gridspec_kw={'wspace': 0.20}
    )
elif len(setups) == 3:
    fig, ax = plt.subplots(
        1, 3,
        figsize=(10.5, 3.2),
        constrained_layout=False,
        gridspec_kw={'wspace': 0.20}
    )
else:
    raise

fields = [q.reshape((n, n)) for q in setups]
vmin = min(q.min() for q in fields)
vmax = max(q.max() for q in fields)

ims = []
for i, (a, q) in enumerate(zip(ax, fields)):
    im = a.imshow(
        q,
        interpolation='bicubic',
        origin='lower',
        extent=extent,
        vmin=vmin,
        vmax=vmax,
        rasterized=True,
    )
    ims.append(im)

    a.set_title(titles[i], pad=6)
    a.set_xlabel(r'$z\;[\mathrm{m}]$')
    a.set_xticks(ticks)
    a.set_yticks(ticks)
    a.set_aspect('equal')

    if i == 0:
        a.set_ylabel(r'$y\;[\mathrm{m}]$')
    else:
        a.set_ylabel('')
        a.tick_params(axis='y', labelleft=False)

cbar = fig.colorbar(ims[0], ax=ax, pad=0.02, fraction=0.046)
#fig.savefig(SAVE_PATH / Path('q_exact_large_inclusions.pdf'), bbox_inches="tight")
plt.show()

In [ ]:
def load_single_experiment(data_dir_path: Path, pattern: re.Pattern) -> Tuple:
    experiment_names = []
    experiment_names += [
        d.name for d in data_dir_path.iterdir()
        if d.is_dir() and pattern.match(d.name)
    ]
    
    data_paths = [data_dir_path / experiment_name for experiment_name in experiment_names]
    file_names = [get_last_file(data_path) for data_path in data_paths]              
    
    setup = None
    data = {}
    optimizer_parameters = {}
    
    for (data_path, file_name) in zip(data_paths, file_names):
        try:
            with open(data_path / file_name, 'rb') as file:
                data_ = load(file)
            data[str(data_path.name)] = data_
        except TypeError:
            print(f"Can not find dumps for {data_path}")
        except:
            print(f"Can not open {data_path / file_name}")
    
        if not setup:
            with open(data_path / 'setup.pkl', 'rb') as file:
                setup = load(file)
    
        
        optimizer_parameter_path = data_path / 'optimizer_parameter.pkl'
        with open(optimizer_parameter_path, 'rb') as file:
            optimizer_parameter = load(file)
            
        optimizer_parameters[str(data_path.name)] = optimizer_parameter
    
    data, FOM_key = filter_and_reorder(data, pattern=r'.*FOM.*')

    return (setup, data)

In [ ]:
data_dir_paths = [
    WORK_DIR / 'elasticity_alu_non_normalize' / 'elasticity_alu_2',
    WORK_DIR / 'elasticity_alu_non_normalize' / 'elasticity_alu_1',
    WORK_DIR / 'elasticity_alu_non_normalize' / 'elasticity_alu_3' 
]

pattern = re.compile(rf'^.*')

data_bundle = {}


for data_dir_path in data_dir_paths:
    
    setup, data = load_single_experiment(
        data_dir_path,
        pattern
    )
    
    data_bundle[data_dir_path.name] = {
        'setup' : setup, 
        'data' : data,
    }

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

from matplotlib.ticker import FormatStrFormatter
from mpl_toolkits.axes_grid1 import make_axes_locatable

plt.rcParams.update({"font.size": 16})

title_pad = 7
time_step = 0
idx = -1

# ------------------------------------------------------------
# Columns = different setups / ground truths
# ------------------------------------------------------------
setup_names = [
    "I",
    "II",
    "III",
]


# Replace this with your actual q_exact sources
bundle_keys = [
    "elasticity_alu_2",
    "elasticity_alu_1",
    "elasticity_alu_3",
]
data_key = 'elasticity_alu_TR_sensors'

# ------------------------------------------------------------
# Grid
# ------------------------------------------------------------
x = np.arange(-30, 31)
y = np.arange(-30, 31)
X, Y = np.meshgrid(x, y)

extent = [-0.5, 0.5, -0.5, 0.5]
ticks = [-0.5, 0.0, 0.5]

# Rows = comparison views
row_titles = [
    r"$q^{\mathsf e}$",
    r"$q^{\textrm{\footnotesize TR}}$",
]

n_rows = 2
n_cols = len(setup_names)

fig = plt.figure(figsize=(3.0 * n_cols, 5.5), constrained_layout=True)
gs = gridspec.GridSpec(
    n_rows, n_cols,
    figure=fig,
    wspace=0.10,
    #hspace=0.08
)

ax = [[fig.add_subplot(gs[i, j]) for j in range(n_cols)] for i in range(n_rows)]

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
for j in range(n_cols):
    q_exact = data_bundle[bundle_keys[j]]["setup"]["q_exact"][time_step, :]
    q_result = data_bundle[bundle_keys[j]]["data"][data_key]["q"][idx].to_numpy()[time_step, :]

    nx, ny = X.shape
    q_exact = q_exact.reshape(nx, ny)
    q_result = q_result.reshape(nx, ny)
    q_diff = q_exact - q_result

    datasets = [q_exact, q_result]

    # same color range for exact + result
    # vmin_shared = min(q_exact.min(), q_result.min())
    # vmax_shared = max(q_exact.max(), q_result.max())

    for i in range(n_rows):
        ax_ = ax[i][j]
        q = datasets[i]

        # if i < 2:
        #     vmin, vmax = vmin_shared, vmax_shared
        # else:
        #     vmin, vmax = q.min(), q.max()

        im = ax_.imshow(
            q,
            origin="lower",
            extent=extent,
            cmap="viridis",
            # vmin=vmin,
            # vmax=vmax,
            aspect="equal",
            interpolation="bilinear"
        )

        ax_.set_xticks(ticks)
        ax_.set_yticks(ticks)

        # column titles
        if i == 0:
            ax_.set_title(setup_names[j], pad=title_pad)

        # row labels on the left only
        if j == 0:
            ax_.set_ylabel(r"$y\;[\mathrm{m}]$")
            ax_.text(
                -0.5, 0.5, row_titles[i],
                transform=ax_.transAxes,
                rotation=90,
                va='center', ha='center'
            )
        else:
            ax_.tick_params(axis="y", labelleft=False)

        # x labels only on bottom row
        if i == n_rows - 1:
            ax_.set_xlabel(r"$z\;[\mathrm{m}]$")
        else:
            ax_.tick_params(axis="x", labelbottom=False)

        # aligned colorbar
        divider = make_axes_locatable(ax_)
        cax_ = divider.append_axes("right", size="3%", pad=0.10)

        cb = fig.colorbar(im, cax=cax_)
        cb.ax.tick_params(labelsize=10, pad=2)
        cb.ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

#plt.show()
fig.savefig(SAVE_PATH / "point_defects_TR_sensors.pdf", bbox_inches="tight")

In [ ]:
data_dir_paths = [
    WORK_DIR / 'elasticity_alu_non_normalize' / 'elasticity_alu_noise_level'
]

pattern = re.compile(rf'^.*')

data_bundle = {}


for data_dir_path in data_dir_paths:
    
    setup, data = load_single_experiment(
        data_dir_path,
        pattern
    )
    
    data_bundle[data_dir_path.name] = {
        'setup' : setup, 
        'data': data,
    }

In [ ]:
data_bundle.keys()

In [ ]:
import math

plt.style.use('default')
plt.rcParams.update({
    "text.usetex": True,          # set to True if you have LaTeX installed
    "font.family": "cm",
    #"font.family": "serif",
    "font.size": 24,
    "axes.labelsize": 24,
    "axes.titlesize": 24,
    "legend.fontsize": 22,
    "xtick.labelsize": 24,
    "ytick.labelsize": 24,
    "figure.dpi": 150,
})


setup_names = [
    r'$\delta_{\mathrm{rel.}} = 0.0$', 
    r'$\delta_{\mathrm{rel.}} = 1.0 \cdot 10^{-3}$', 
    r'$\delta_{\mathrm{rel.}} = 5.0 \cdot 10^{-3}$', 
    r'$\delta_{\mathrm{rel.}} = 1.0 \cdot 10^{-2}$', 
]

bundle_keys = ['elasticity_alu_noise_level']
data_keys = [
    'elasticity_alu_TR_sensors_noise_level_0.0', 
    'elasticity_alu_TR_sensors_noise_level_0.001', 
    'elasticity_alu_TR_sensors_noise_level_0.005', 
    'elasticity_alu_TR_sensors_noise_level_0.01', 

]

n_total = len(setup_names)
n_rows = 1
n_cols = math.ceil(n_total / n_rows)

fig = plt.figure(figsize=(4.5 * n_cols, 8.0), constrained_layout=True)
gs = gridspec.GridSpec(
    n_rows, n_cols,
    figure=fig,
    wspace=0.07,
    hspace=0.03,
)

ax = [[fig.add_subplot(gs[i, j]) for j in range(n_cols)] for i in range(n_rows)]

# ------------------------------------------------------------
# Plot
# ------------------------------------------------------------
for k in range(n_total):
    i = k // n_cols   # row
    j = k % n_cols    # column

    ax_ = ax[i][j]

    q_result = data_bundle[bundle_keys[0]]["data"][data_keys[k]]["q"][idx].to_numpy()[time_step, :]

    nx, ny = X.shape
    q_result = q_result.reshape(nx, ny)

    im = ax_.imshow(
        q_result,
        origin="lower",
        extent=extent,
        cmap="viridis",
        aspect="equal",
        interpolation="bilinear"
    )

    ax_.set_xticks(ticks)
    ax_.set_yticks(ticks)

    # titles
    ax_.set_title(setup_names[k], pad=title_pad)

    # y labels only on first column
    if j == 0:
        ax_.set_ylabel(r"$y\;[\mathrm{m}]$")
    else:
        ax_.tick_params(axis="y", labelleft=False)

    # x labels on the lowest occupied subplot in each column
    has_plot_below = ((i + 1) * n_cols + j) < n_total
    if not has_plot_below:
        ax_.set_xlabel(r"$z\;[\mathrm{m}]$")
    else:
        ax_.tick_params(axis="x", labelbottom=False)

    # colorbar
    divider = make_axes_locatable(ax_)
    cax_ = divider.append_axes("right", size="3%", pad=0.10)

    cb = fig.colorbar(im, cax=cax_)
    cb.ax.tick_params(labelsize=20, pad=2)
    cb.ax.yaxis.set_major_formatter(FormatStrFormatter("%.2f"))

# hide unused subplot (last one)
if n_total < n_rows * n_cols:
    for k in range(n_total, n_rows * n_cols):
        i = k // n_cols
        j = k % n_cols
        ax[i][j].axis("off")

plt.subplots_adjust(
    left=0.06,
    right=0.98,
    bottom=0.06,
    top=0.94,
    wspace=0.02,
    hspace=0.02,
)
plt.savefig(SAVE_PATH / "noise_level_q_TRs.pdf", bbox_inches="tight")